In [0]:
%sql
use catalog datastore

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS datastore.silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS QUARANTINE

In [0]:
# =====================================================================
# Silver ETL (PySpark): datastore.bronze.transactions -> datastore.silver.transactions
#   Hard validation failures are routed to datastore.quarantine.transactions with a
#   reason code (never silently dropped).
#   Soft issues (non-INR currency, negative/zero amount, date/timestamp
#   mismatch, future-dated rows, unparseable is_refund, dimension keys not
#   found) stay in silver with review flags instead of being quarantined.
#   Dimension keys are matched case/whitespace-insensitively so formatting
#   differences don't create false orphans.
#   NOTE: adjust dimension table/column names if your bronze schema differs.
# =====================================================================
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG = "datastore"

# --- generic column standardization --------------------------------------
# 1) Every STRING column is lowercased (and trimmed).
# 2) Any column whose text parses as a date or datetime is rewritten to a
#    canonical format: 'yyyy-MM-dd' for dates, 'yyyy-MM-dd HH:mm:ss' for
#    datetimes. Values that don't parse as dates keep their lowercased text,
#    so free-text columns are unaffected.
from pyspark.sql.types import StringType

def standardize_dataframe(df):
    """Lowercase all string columns and standardize all date columns."""
    select_exprs = []
    for field in df.schema.fields:
        col_ref = f"`{field.name}`"
        if isinstance(field.dataType, StringType):
            col_str = F.col(col_ref)
            # does the text carry a time-of-day component?
            has_time = col_str.rlike(r"\d{1,2}:\d{2}")
            # datetime: try common formats (null when not parseable)
            as_ts = F.coalesce(
                F.expr(f"try_to_timestamp({col_ref}, 'yyyy-MM-dd HH:mm:ss')"),
                F.expr(f"try_to_timestamp({col_ref}, 'dd/MM/yyyy HH:mm:ss')"),
                F.expr(f"try_to_timestamp({col_ref}, 'dd-MM-yyyy HH:mm:ss')"),
                F.expr(f"try_to_timestamp({col_ref}, 'MM/dd/yyyy HH:mm:ss')"),
                F.expr(f"try_to_timestamp({col_ref}, 'dd-MMM-yyyy HH:mm:ss')"),
                F.expr(f"try_to_timestamp({col_ref}, 'yyyy-MM-dd HH:mm')"),
                F.expr(f"try_to_timestamp({col_ref}, 'dd/MM/yyyy HH:mm')"),
                F.expr(f"try_to_timestamp({col_ref}, 'yyyy/MM/dd HH:mm:ss')"),
                F.expr(f"try_to_timestamp({col_ref}, 'MM/dd/yyyy HH:mm')"),
            )
            # date: try common formats (null when not parseable)
            as_date = F.coalesce(
                F.expr(f"try_to_date({col_ref}, 'yyyy-MM-dd')"),
                F.expr(f"try_to_date({col_ref}, 'dd/MM/yyyy')"),
                F.expr(f"try_to_date({col_ref}, 'dd-MM-yyyy')"),
                F.expr(f"try_to_date({col_ref}, 'MM/dd/yyyy')"),
                F.expr(f"try_to_date({col_ref}, 'dd-MMM-yyyy')"),
                F.expr(f"try_to_date({col_ref}, 'yyyyMMdd')"),
                F.expr(f"try_to_date({col_ref}, 'yyyy/MM/dd')"),
                F.expr(f"try_to_date({col_ref}, 'MM-dd-yyyy')"),
            )
            select_exprs.append(
                F.when(has_time & as_ts.isNotNull(),
                       F.date_format(as_ts, "yyyy-MM-dd HH:mm:ss"))
                .when(~has_time & as_date.isNotNull(),
                      F.date_format(as_date, "yyyy-MM-dd"))
                .otherwise(F.lower(F.trim(col_str)))
                .alias(field.name)
            )
        else:
            select_exprs.append(F.col(col_ref))
    return df.select(*select_exprs)

base = standardize_dataframe(spark.table(f"{CATALOG}.bronze.transactions"))

# --- transaction_ids that map to more than one distinct payload ---------
id_conflicts = (
    base.where(F.col("transaction_id").isNotNull())
        .groupBy("transaction_id")
        .agg(F.expr("count(distinct to_json(struct(*)))").alias("payload_variants"))
        .where(F.col("payload_variants") > 1)
        .select("transaction_id", F.lit(True).alias("_is_conflicting_id"))
)

# --- standardize / parse each bronze row --------------------------------
pm = F.trim(F.col("payment_method").cast("string"))
st = F.trim(F.col("status").cast("string"))
ir = F.lower(F.trim(F.col("is_refund").cast("string")))
amt_str = F.regexp_replace(F.trim(F.col("amount").cast("string")), "[^0-9.-]", "")

typed = base.select(
    "*",
    # amount: strip currency symbols, commas and spaces; blank -> NULL
    F.when(F.length(amt_str) > 0, amt_str.cast("decimal(18,2)")).alias("parsed_amount"),
    # timestamp: try multiple formats
    F.expr("""
        coalesce(
          try_to_timestamp(cast(`timestamp` as string)),
          try_to_timestamp(cast(`timestamp` as string), 'dd/MM/yyyy HH:mm:ss'),
          try_to_timestamp(cast(`timestamp` as string), 'dd-MM-yyyy HH:mm:ss'),
          try_to_timestamp(cast(`timestamp` as string), 'MM/dd/yyyy HH:mm:ss'),
          try_to_timestamp(cast(`timestamp` as string), 'dd-MMM-yyyy HH:mm:ss'),
          try_to_timestamp(cast(`timestamp` as string), 'yyyy/MM/dd HH:mm:ss'),
          try_to_timestamp(cast(`timestamp` as string), 'yyyy-MM-dd')
        )
    """).alias("parsed_ts"),
    # date: try multiple formats
    F.expr("""
        coalesce(
          try_to_date(cast(`date` as string), 'yyyy-MM-dd'),
          try_to_date(cast(`date` as string), 'dd/MM/yyyy'),
          try_to_date(cast(`date` as string), 'dd-MM-yyyy'),
          try_to_date(cast(`date` as string), 'MM/dd/yyyy'),
          try_to_date(cast(`date` as string), 'dd-MMM-yyyy'),
          try_to_date(cast(`date` as string), 'yyyyMMdd'),
          try_to_date(cast(`date` as string), 'yyyy/MM/dd'),
          try_to_date(cast(`date` as string), 'MM-dd-yyyy')
        )
    """).alias("parsed_date"),
    F.upper(F.trim(F.col("currency").cast("string"))).alias("std_currency"),
    # payment_method: map common variants to a single canonical form
    (
        F.when(F.col("payment_method").isNull(), F.lit(None).cast("string"))
        .when(pm.rlike("(?i)credit"), F.lit("CREDIT_CARD"))
        .when(pm.rlike("(?i)debit"), F.lit("DEBIT_CARD"))
        .when(pm.rlike("(?i)upi"), F.lit("UPI"))
        .when(pm.rlike("(?i)net[ _-]?bank"), F.lit("NET_BANKING"))
        .when(pm.rlike("(?i)wallet"), F.lit("WALLET"))
        .when(pm.rlike("(?i)neft"), F.lit("NEFT"))
        .when(pm.rlike("(?i)rtgs"), F.lit("RTGS"))
        .when(pm.rlike("(?i)cheque"), F.lit("CHEQUE"))
        .when(pm.rlike("(?i)cash"), F.lit("CASH"))
        .when(pm.rlike("(?i)card"), F.lit("CARD"))
        .otherwise(F.upper(F.regexp_replace(pm, "[^A-Za-z0-9]+", "_")))
    ).alias("std_payment_method"),
    # status: map common variants to canonical values
    (
        F.when(F.col("status").isNull(), F.lit(None).cast("string"))
        .when(st.rlike("(?i)complet|success"), F.lit("COMPLETED"))
        .when(st.rlike("(?i)fail|reject|declin"), F.lit("FAILED"))
        .when(st.rlike("(?i)pending|process|init"), F.lit("PENDING"))
        .when(st.rlike("(?i)cancel"), F.lit("CANCELLED"))
        .when(st.rlike("(?i)refund"), F.lit("REFUNDED"))
        .otherwise(F.upper(st))
    ).alias("std_status"),
    # is_refund: normalize Y/N, yes/no, true/false, 1/0
    # unparseable/missing values default to False (not a refund) and are
    # flagged for review instead of quarantining the row
    (
        F.when(ir.isin("y", "yes", "true", "1"), F.lit(True))
        .when(ir.isin("n", "no", "false", "0"), F.lit(False))
        .otherwise(F.lit(False))
    ).alias("std_is_refund"),
    (~F.coalesce(ir.isin("y", "yes", "true", "1", "n", "no", "false", "0"), F.lit(False)))
    .alias("is_refund_needs_review"),
)

# --- referential integrity against Customer / Branch / Product ----------
def dim_keys(table: str, key: str, marker: str):
    return (
        spark.table(f"{CATALOG}.bronze.{table}")
        .where(F.col(key).isNotNull())
        # standardize dimension keys the same way the fact rows are standardized
        # so casing/whitespace differences don't produce false orphan matches
        .select(F.lower(F.trim(F.col(key).cast("string"))).alias(f"dim_{key}"))
        .distinct()
        .withColumn(marker, F.lit(True))
    )

refchecked = (
    typed
    .join(id_conflicts, "transaction_id", "left")
    .join(
        dim_keys("customers", "account_id", "_acct_found"),
        F.lower(F.trim(F.col("account_id").cast("string"))) == F.col("dim_account_id"),
        "left",
    )
    .join(
        dim_keys("branches", "branch_id", "_branch_found"),
        F.lower(F.trim(F.col("branch_id").cast("string"))) == F.col("dim_branch_id"),
        "left",
    )
    .join(
        dim_keys("products", "product_id", "_prod_found"),
        F.lower(F.trim(F.col("product_id").cast("string"))) == F.col("dim_product_id"),
        "left",
    )
    .withColumn("is_conflicting_id", F.coalesce(F.col("_is_conflicting_id"), F.lit(False)))
    .withColumn("account_found", F.coalesce(F.col("_acct_found"), F.lit(False)))
    .withColumn("branch_found", F.coalesce(F.col("_branch_found"), F.lit(False)))
    .withColumn("product_found", F.coalesce(F.col("_prod_found"), F.lit(False)))
    .drop(
        "_is_conflicting_id", "_acct_found", "_branch_found", "_prod_found",
        "dim_account_id", "dim_branch_id", "dim_product_id",
    )
)

# --- exact duplicate rows (identical across every column): keep one ----
row_json = F.to_json(F.struct(*[F.col(c) for c in refchecked.columns]))
exact_dup_window = Window.partitionBy(row_json).orderBy(row_json)
deduped = refchecked.withColumn("exact_dup_rn", F.row_number().over(exact_dup_window))

# --- reconcile date vs timestamp + review flags + reason codes ----------
std_transaction_date = F.coalesce(F.col("parsed_ts").cast("date"), F.col("parsed_date"))

txn_stage = (
    deduped
    # trust the timestamp when date and timestamp disagree
    .withColumn("std_transaction_date", std_transaction_date)
    # soft review flags (rows stay in silver)
    .withColumn(
        "date_ts_mismatch",
        F.col("parsed_date").isNotNull() & F.col("parsed_ts").isNotNull()
        & (F.col("parsed_date") != F.col("parsed_ts").cast("date")),
    )
    .withColumn(
        "currency_needs_review",
        F.col("std_currency").isNotNull() & (F.col("std_currency") != "INR"),
    )
    .withColumn(
        "is_negative_amount",
        F.col("parsed_amount").isNotNull() & (F.col("parsed_amount") < 0),
    )
    .withColumn(
        "is_zero_amount",
        F.col("parsed_amount").isNotNull() & (F.col("parsed_amount") == 0),
    )
    .withColumn("is_future_dated", std_transaction_date > F.current_date())
    # soft review flags for keys not found in the dimension tables (rows stay
    # in silver; the reference can be repaired later)
    .withColumn("account_id_needs_review", ~F.col("account_found"))
    .withColumn("branch_id_needs_review", ~F.col("branch_found"))
    .withColumn("product_id_needs_review", ~F.col("product_found"))
    # hard failure reason codes -> quarantine (multiple codes joined by ';')
    .withColumn(
        "reason_code",
        F.concat_ws(
            ";",
            F.when(F.col("transaction_id").isNull(), F.lit("MISSING_TRANSACTION_ID")),
            F.when(F.col("is_conflicting_id"), F.lit("CONFLICTING_TRANSACTION_ID")),
            F.when(F.col("exact_dup_rn") > 1, F.lit("EXACT_DUPLICATE")),
            F.when(F.col("parsed_amount").isNull(), F.lit("INVALID_AMOUNT")),
            F.when(F.col("parsed_ts").isNull() & F.col("parsed_date").isNull(), F.lit("INVALID_DATE")),
        ),
    )
)

# --- Rejects path: every record that failed validation ------------------
(
    txn_stage.where(F.col("reason_code") != "")
    .withColumn("quarantined_at", F.current_timestamp())
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.quarantine.transactions")
)

# --- Clean path: rows with no hard failures -----------------------------
(
    txn_stage.where(F.col("reason_code") == "")
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.transactions")
)

# --- Run summary --------------------------------------------------------
silver_tbl = spark.table(f"{CATALOG}.silver.transactions")
quarantine_tbl = spark.table(f"{CATALOG}.quarantine.transactions")

silver_stats = silver_tbl.agg(
    F.count(F.lit(1)).alias("silver_rows"),
    F.count(F.when(F.col("currency_needs_review"), F.lit(1))).alias("non_inr_rows"),
    F.count(F.when(F.col("date_ts_mismatch"), F.lit(1))).alias("date_ts_mismatch_rows"),
    F.count(F.when(F.col("is_future_dated"), F.lit(1))).alias("future_dated_rows"),
    F.count(
        F.when(
            F.col("account_id_needs_review")
            | F.col("branch_id_needs_review")
            | F.col("product_id_needs_review"),
            F.lit(1),
        )
    ).alias("orphan_key_rows"),
    F.count(F.when(F.col("is_refund_needs_review"), F.lit(1))).alias("is_refund_defaulted_rows"),
).first()

quarantine_stats = quarantine_tbl.agg(
    F.count(F.lit(1)).alias("quarantined_rows"),
    F.count(
        F.when(F.col("is_negative_amount") | F.col("is_zero_amount"), F.lit(1))
    ).alias("amount_review_rows"),
).first()

display(
    spark.createDataFrame(
        [(
            silver_stats["silver_rows"],
            quarantine_stats["quarantined_rows"],
            silver_stats["non_inr_rows"],
            silver_stats["date_ts_mismatch_rows"],
            silver_stats["future_dated_rows"],
            quarantine_stats["amount_review_rows"],
            silver_stats["orphan_key_rows"],
            silver_stats["is_refund_defaulted_rows"],
        )],
        schema="""
            silver_rows long, quarantined_rows long, non_inr_rows long,
            date_ts_mismatch_rows long, future_dated_rows long, amount_review_rows long,
            orphan_key_rows long, is_refund_defaulted_rows long
        """,
    )
)

In [0]:
%sql
select * from silver.transactions

In [0]:
%sql
select * from quarantine.transactions
